# Cox Proportional Hazards
        **tabular › survival** · notebook `cox_ph`

        **Kapan dipakai:** Model survival standar: rasio hazard tiap fitur dapat dibaca (bahan laporan: 'usia lebih muda meningkatkan risiko x%').

        **Jebakan umum:** Asumsi hazard proporsional (periksa dengan uji asumsi); fitur korelasi tinggi tidak stabil (gunakan penalizer).

**Data demo:** Rossi (432 narapidana; durasi sampai ditangkap lagi, 74% tersensor). **Tersensor** = pengamatan berakhir tanpa kejadian (mis. belum ditangkap lagi sampai akhir studi): informasinya 'bertahan setidaknya sampai waktu ini', bukan hilang.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular_extra/survival/rossi_train.csv"; TEST_PATH = ROOT / "data_latihan/tabular_extra/survival/rossi_test.csv"
TEST_LABELS = TEST_PATH.with_name(TEST_PATH.name.replace("_test.csv", "_test_labels.csv"))      # durasi/kejadian uji (hanya latihan)
DURATION, EVENT = "week", "arrest"
FEATURES = ['fin', 'age', 'race', 'wexp', 'mar', 'paro', 'prio']
MODEL_KEY = "cox_ph"; SEED = 42
OUT = out_dir("tabular", "survival", MODEL_KEY); set_seed(SEED)

## 2. Data: durasi, kejadian, dan sensor
**Jangan** memperlakukan baris tersensor sebagai 'tidak ada kejadian' lalu memakai regresi/klasifikasi biasa: itu membuang informasi dan bias.

In [ ]:
from types import SimpleNamespace
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH).merge(pd.read_csv(TEST_LABELS), on="id")
print(f"latih {len(train)} | uji {len(test)} | kejadian pada latih: {int(train[EVENT].sum())} ({train[EVENT].mean():.0%}), sisanya tersensor ({1 - train[EVENT].mean():.0%})")
km = KaplanMeierFitter().fit(train[DURATION], train[EVENT]); ax = km.plot_survival_function(figsize=(6, 3.2)); ax.set_title("Kurva kelangsungan hidup keseluruhan (Kaplan-Meier)")
plt.tight_layout(); plt.savefig(OUT / "km_overall.png", dpi=130); plt.show()
print("median kelangsungan hidup:", km.median_survival_time_)

## 3. Model
Semua model berbentuk `fit_predict(train, test, ctx)` yang mengembalikan **skor risiko** (lebih besar = risiko kejadian lebih tinggi/waktu lebih pendek).

In [ ]:
import numpy as np, pandas as pd
from lifelines import CoxPHFitter

def fit_predict(train, test, ctx):
    m = CoxPHFitter(penalizer=0.1).fit(train[ctx.features + [ctx.duration, ctx.event]], ctx.duration, ctx.event)
    ctx.state['model'] = m
    return m.predict_partial_hazard(test[ctx.features]).to_numpy()

## 4. Evaluasi: **concordance index (C-index)**
Dari semua pasangan pasien yang urutannya bisa dipastikan, berapa persen yang diurutkan benar oleh skor risiko. 0.5 = acak, 1 = sempurna. Sampel uji kecil → ketidakpastian besar; kami beri selang kepercayaan bootstrap.

In [ ]:
ctx = SimpleNamespace(features=FEATURES, duration=DURATION, event=EVENT, state={})
import time; t0 = time.time(); risk = np.asarray(fit_predict(train, test, ctx), float); sec = time.time() - t0
ci = concordance_index(test[DURATION], -risk, test[EVENT])
rng = np.random.default_rng(SEED); boots = []
for _ in range(300):
    b = rng.integers(0, len(test), len(test))
    try: boots.append(concordance_index(test[DURATION].to_numpy()[b], -risk[b], test[EVENT].to_numpy()[b]))
    except ZeroDivisionError: pass
lo, hi = np.percentile(boots, [2.5, 97.5]); metrics = {"c_index_test": float(ci), "ci95_low": float(lo), "ci95_high": float(hi), "seconds": sec}
display(show_table(metrics)); print(f"C-index uji = {ci:.3f} (95% CI {lo:.3f}-{hi:.3f}); acak = 0.5")

## 5. Apakah skor risiko memisahkan pasien? (kurva KM per kuartil risiko, data uji)
Kurva kuartil risiko tertinggi harus turun lebih cepat daripada kuartil terendah.

In [ ]:
q = pd.qcut(pd.Series(risk).rank(method="first"), 4, labels=["Q1 risiko rendah", "Q2", "Q3", "Q4 risiko tinggi"]).to_numpy()
fig, ax = plt.subplots(figsize=(6, 3.4))
for g_ in ["Q1 risiko rendah", "Q2", "Q3", "Q4 risiko tinggi"]:
    m_ = q == g_; KaplanMeierFitter().fit(test.loc[m_, DURATION], test.loc[m_, EVENT], label=f"{g_} (n={int(m_.sum())})").plot_survival_function(ax=ax, ci_show=False)
ax.set_title("Kelangsungan hidup uji menurut kuartil risiko"); plt.tight_layout(); plt.savefig(OUT / "risk_groups.png", dpi=130); plt.show()

## 6. Tafsir model

In [ ]:
m = ctx.state.get("model")
if hasattr(m, "summary"):
    s = m.summary.reset_index(); display(s.iloc[:, [i for i, c in enumerate(s.columns) if c in ("param", "covariate", "coef", "exp(coef)", "p")]].round(3).head(12))
    print("exp(coef) > 1: fitur menaikkan hazard (Cox) / memperpanjang waktu (AFT: exp(coef) > 1 = waktu lebih lama)")
elif m is not None:
    imp = pd.Series(m.get_score(importance_type="gain") if hasattr(m, "get_score") else dict(zip(FEATURES, m.feature_importances_)))
    imp = imp.rename(index=lambda k: FEATURES[int(k[1:])] if str(k).startswith("f") and str(k)[1:].isdigit() else k); display(imp.sort_values(ascending=False).round(3).to_frame("importance (gain)"))

In [ ]:
pd.DataFrame({"id": test["id"], "risk": risk}).to_csv(OUT / "risk_scores.csv", index=False)
save_json({"metrics": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan Cox, AFT, dan XGBoost lewat C-index dengan selang kepercayaan: selisih kecil pada sampel kecil tidak bermakna.
- Periksa asumsi hazard proporsional (`cph.check_assumptions`) sebelum menafsirkan rasio hazard.
- Prediksi probabilitas bertahan pada waktu tertentu: `predict_survival_function`.